first: get access to the parquet
second: put all genres and themes into a set, creating a vocabulary for matrix later
third: sort set, and map out all tags to an index/column number
fourth: make a matrix, 4812 x 73 size, which is # of anime x # of tags. all values are 0
fifth: iterate through every anime, and for their corresponding genres/themes, make those values 1 in the matrix

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet(path="../data/anime.parquet")

vocabulary = set()

# for every anime, add the genres and themes into a set, to create the vocabulary needed for the vector matrix
for a in df["genres"]:
    for genre in a:
        vocabulary.add(genre)
for a in df["themes"]:
    for theme in a:
        vocabulary.add(theme)


sorted_vocab = sorted(vocabulary)


# create dict mapping to all different genres/themes
tag_map = {}
for n, t in enumerate(sorted_vocab):
    tag_map[t] = n

# make matrix
matrix = np.zeros((len(df),len(sorted_vocab)), dtype=np.float32)

# every anime ordered from 0-4800 approx
for i in range(len(df["title"])):
    genres = df["genres"][i] # get the genres, by accessing the genres at that specific index, returns a list
    # iterate the list of genres for said anime, then in the matrix, map to the specific anime and then tag_map to find the column of that tag
    for g in genres:
        matrix[i][tag_map[g]] = 1
    themes = df["themes"][i]
    for t in themes:
        matrix[i][tag_map[t]] = 1

def find_title(name: str):
    return df[df["title"].str.contains(name, case=False, na=False)]


title_to_row = {}
for i, id in enumerate(df["title"]):
    title_to_row[id] = i

# prefs example (3 inputs)
rows = []
inputs = ["Jujutsu Kaisen", "Attack on Titan Season 2", "Blue Lock", "Death Note"]
for t in inputs:
    rows.append(matrix[title_to_row[t]])

taste_vector = np.mean(rows, axis=0) # just adding all separate vectors together with their corresponding section, then averaging by # of inputs


# cosine similarity formula

# does dot product between matrix and taste_vector, as well as the sum, so every value is now similarity btw prefs and said anime
numerators = matrix @ taste_vector

# denominators np func basically just finds vector length using sqrt(sum of squares), we find for both and we multiply em 
taste_norm = np.linalg.norm(taste_vector)
anime_norms = np.linalg.norm(matrix, axis=1) # matrix is a list of the anime vector embeddings, so each one has a different vector length

# division
scores = numerators / (taste_norm * anime_norms)



# print out recs

order = np.argsort(scores)[::-1] # gives us index that gave us the highest scores

count = 0
for index in order:
    if count >= 10:
        break
    title = df["title"].iloc[index]
    if title in inputs:
        continue
    print(title)
    count+=1

    




Attack on Titan
The Future Diary
Attack on Titan: The Last Attack
Attack on Titan: Final Season
Attack on Titan Season 3
Attack on Titan Season 3 Part 2
Tekkonkinkreet
Attack on Titan: Chronicle
Attack on Titan: Final Season Part 2
Attack on Titan: Final Season - The Final Chapters


In [2]:
import re

# ---- title-normalization hack: collapse an anime title down to a "franchise key" ----
# NOTE: this is a heuristic, not the real fix. The proper fix uses MAL's relations graph.
# It just strips season/part/type/subtitle noise so all "Attack on Titan ..." entries
# reduce to the same key ("attack on titan") and we can skip duplicates from one franchise.
def franchise_key(title: str) -> str:
    key = title.lower()
    key = key.split(":")[0]                     # drop subtitle after a colon (": Final Season")
    # remove common season / part / type words
    key = re.sub(r"\b(season|part|cour|final|the final chapters?|movie|film|ova|ona|special|tv)\b", " ", key)
    key = re.sub(r"\b[ivx]+\b", " ", key)       # roman numerals (ii, iii, iv, ...)
    key = re.sub(r"\d+", " ", key)              # arabic numbers (3, 2, ...)
    key = re.sub(r"[^a-z0-9]+", " ", key)       # punctuation -> space
    key = re.sub(r"\s+", " ", key).strip()      # collapse whitespace
    return key


# seed the "already seen" set with the franchises of the anime you input,
# so we don't recommend a different season of something you already listed.
seen_franchises = {franchise_key(t) for t in inputs}

print("=== deduped recommendations ===")
count = 0
for index in order:
    if count >= 10:
        break
    title = df["title"].iloc[index]
    if title in inputs:
        continue

    key = franchise_key(title)
    if key in seen_franchises:   # already showed something from this franchise -> skip
        continue
    seen_franchises.add(key)

    print(f"{scores[index]:.3f}  {title}")
    count += 1


=== deduped recommendations ===
0.742  The Future Diary
0.713  Tekkonkinkreet
0.683  Attack on Titan OAD
0.683  Ga-Rei-Zero
0.660  Deadman Wonderland
0.655  Wonder Egg Priority
0.655  Lookism
0.630  Jujutsu Kaisen Season 2 Recaps
0.630  Rozen Maiden
0.630  Beyond the Boundary
